# 03 - Cari kandidat komentar negatif tambahan (Ollama: bge-m3 + qwen2.5:7b)

Tujuan: dari 28.711 komentar mentah di `comments.csv` (belum dilabel), cari kandidat yang **kemungkinan negatif**, supaya kita punya lebih banyak contoh negatif buat training (saat ini cuma ~49 dari 2000).

Dua lapis, biar nggak harus nge-judge 28rb komentar satu-satu pakai LLM (lama):

- **Layer 1 — semantic search (`bge-m3`)**: hitung kemiripan makna tiap komentar ke 49 komentar negatif yang sudah ada. Cepat, motong 28.711 jadi `N_CANDIDATES` teratas.
- **Layer 2 — LLM judge (`qwen2.5:7b`)**: tiap kandidat dari Layer 1 di-tanya langsung pakai definisi yang sudah kita sepakati ("negatif = nyindir/kritik ke konten atau dokter, BUKAN curhat pribadi"), lengkap dengan contoh asli dari data labelmu (few-shot, bukan contoh karangan).

**Hasil akhir tetap kamu cek manual** — notebook ini cuma mempersempit 28.711 jadi daftar kandidat yang masuk akal buat direview.

**Ketahanan proses (biar aman kalau timeout/ke-interrupt):**
- Setiap embedding yang berhasil **langsung ditulis ke disk** (bukan ditahan di memory sampai akhir) -- jadi kalau proses berhenti di tengah, yang sudah kehitung nggak hilang.
- Kalau notebook dijalankan ulang, dia **baca dulu cache yang sudah ada** dan cuma embed sisa yang belum, bukan mulai dari nol.
- Semua hasil (cache, kandidat, dll) disimpan di folder `hasil/`.

**Prasyarat:**
- Ollama jalan di komputer kamu (`ollama serve`, biasanya otomatis jalan di background)
- Model sudah ke-pull: `bge-m3`, `qwen2.5:7b` (sudah ada di `ollama list` kamu)
- `pip install requests pandas numpy tqdm`

In [8]:
import csv
import hashlib
import json
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm

# ============================ KONFIGURASI ============================
OLLAMA_HOST    = 'http://localhost:11434'
EMBED_MODEL    = 'bge-m3'
JUDGE_MODEL    = 'qwen2.5:7b'
N_CANDIDATES   = 5000     # berapa kandidat Layer 1 yang di-judge LLM -- besarin kalau mau lebih teliti, tapi makin lama
N_FEWSHOT_EACH = 4        # jumlah contoh negatif & non-negatif asli yang disisipkan ke prompt judge
CHECKPOINT_EVERY = 100    # Layer 2: simpan progres tiap N komentar
SEED           = 42
# =====================================================================

# Notebook ini ditaruh di: 01_pengumpulan_data\06_data_negatif_tambahan\
# comments.csv ditaruh di folder yang SAMA dengan notebook ini.
# dataset_final.csv (hasil 01_persiapan_data.ipynb) ada di folder terpisah: 02_IndoBert\data\siap_latih\binary\
NOTEBOOK_DIR = Path.cwd()
REPO_ROOT    = NOTEBOOK_DIR.parent.parent   # 06_data_negatif_tambahan -> 01_pengumpulan_data -> root repo

LABELED_PATH = REPO_ROOT / '02_IndoBert' / 'data' / 'siap_latih' / 'binary' / 'dataset_final.csv'
RAW_PATH     = NOTEBOOK_DIR / 'comments.csv'    # 28.711 komentar mentah -- taruh di folder yang sama dgn notebook ini

OUT_DIR      = NOTEBOOK_DIR / 'hasil'
OUT_DIR.mkdir(parents=True, exist_ok=True)

EMBED_CACHE_REF_PATH = OUT_DIR / 'embed_cache_acuan_negatif.csv'   # embedding 49 komentar acuan, per-baris langsung ditulis
EMBED_CACHE_RAW_PATH = OUT_DIR / 'embed_cache_comments.csv'        # embedding comments.csv, per-baris langsung ditulis
LAYER1_PATH          = OUT_DIR / 'layer1_kandidat_embedding.csv'
FINAL_PATH           = OUT_DIR / 'kandidat_final_untuk_review.csv'

for p in [LABELED_PATH, RAW_PATH]:
    print(('OK      ' if p.exists() else 'HILANG  ') + str(p))

# cek Ollama hidup + model yang dibutuhkan sudah ke-pull
try:
    tags = requests.get(f'{OLLAMA_HOST}/api/tags', timeout=5).json()
    available = {m['name'] for m in tags.get('models', [])}
    print('Ollama hidup. Model tersedia:', available)
    for m in [EMBED_MODEL, JUDGE_MODEL]:
        hit = any(m == a or a.startswith(m + ':') or m.startswith(a.split(':')[0]) for a in available)
        ok_str = 'OK' if hit else 'TIDAK DITEMUKAN, cek nama model / ollama pull'
        print(f'  {m:<14} -> {ok_str}')
except Exception as e:
    print('Ollama belum jalan / tidak bisa diakses:', e)
    print('Jalankan "ollama serve" dulu di terminal, lalu run ulang cell ini.')

OK      c:\Users\nesha\Tugas_akhir\seleksi-influencer\02_IndoBert\data\siap_latih\binary\dataset_final.csv
OK      c:\Users\nesha\Tugas_akhir\seleksi-influencer\01_pengumpulan_data\06_data_negatif_tambahan\comments.csv
Ollama hidup. Model tersedia: {'qwen2.5:7b', 'bge-m3:latest', 'llama3.1:latest'}
  bge-m3         -> OK
  qwen2.5:7b     -> OK


## Fungsi panggil Ollama (embedding & judge) + cache embedding yang aman dari timeout

`embed_with_cache()`: untuk sekumpulan teks, baca dulu cache yang sudah ada di disk (resume), lalu
cuma embed sisanya. Tiap satu teks berhasil di-embed, langsung ditulis 1 baris ke file CSV cache dan
di-`flush()` -- jadi aman kalau prosesnya ke-interrupt di tengah jalan, nggak ada yang hilang.

In [9]:
def get_embedding(text: str, model: str = EMBED_MODEL, retries: int = 3) -> np.ndarray:
    for attempt in range(retries):
        try:
            r = requests.post(f'{OLLAMA_HOST}/api/embeddings',
                               json={'model': model, 'prompt': text}, timeout=30)
            r.raise_for_status()
            return np.array(r.json()['embedding'], dtype=np.float32)
        except Exception as e:
            if attempt == retries - 1:
                raise
            time.sleep(1.5 * (attempt + 1))


def cosine_sim_matrix(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    a_n = a / (np.linalg.norm(a, axis=1, keepdims=True) + 1e-8)
    b_n = b / (np.linalg.norm(b, axis=1, keepdims=True) + 1e-8)
    return a_n @ b_n.T


def _hash_text(t: str) -> str:
    return hashlib.md5(t.encode('utf-8')).hexdigest()


def _load_embed_cache(path: Path) -> dict:
    """hash -> (text, vektor). Dibaca dari file CSV cache kalau ada."""
    cache = {}
    if not path.exists():
        return cache
    with open(path, 'r', encoding='utf-8', newline='') as f:
        reader = csv.reader(f)
        header = next(reader, None)
        for row in reader:
            if len(row) < 3:
                continue
            h, text, emb_json = row[0], row[1], row[2]
            try:
                cache[h] = (text, np.array(json.loads(emb_json), dtype=np.float32))
            except Exception:
                continue   # baris cache korup (misal ke-cut pas crash) -- dilewati, bukan nge-gagalin semua
    return cache


def embed_with_cache(texts: list, cache_path: Path, desc: str) -> dict:
    """Return dict {text: vektor}. Resume dari cache_path; tiap embedding baru yang berhasil
    langsung ditulis+flush ke cache_path, jadi aman kalau proses ke-interrupt."""
    cache = _load_embed_cache(cache_path)
    print(f'  cache ditemukan: {len(cache)} embedding tersimpan sebelumnya ({cache_path.name})')

    result = {text: vec for (text, vec) in cache.values()}
    already_hashed = set(cache.keys())
    todo = [t for t in texts if _hash_text(t) not in already_hashed]
    print(f'  sudah di-embed sebelumnya: {len(texts) - len(todo)} dari {len(texts)}')
    print(f'  perlu embed baru         : {len(todo)}')

    if not todo:
        return result

    is_new_file = not cache_path.exists()
    f = open(cache_path, 'a', encoding='utf-8', newline='')
    writer = csv.writer(f)
    if is_new_file:
        writer.writerow(['hash', 'text', 'embedding_json'])
        f.flush()
    try:
        for t in tqdm(todo, desc=desc):
            try:
                vec = get_embedding(t)
            except Exception as e:
                print('Gagal embed (dilewati, akan dicoba lagi kalau notebook di-run ulang):',
                      repr(t)[:80], '->', e)
                continue
            result[t] = vec
            writer.writerow([_hash_text(t), t, json.dumps(vec.tolist())])
            f.flush()   # <- ini kuncinya: langsung masuk disk, nggak nunggu loop selesai
    finally:
        f.close()
    return result


JUDGE_PROMPT_TEMPLATE = """Kamu menilai apakah komentar TikTok di bawah ini termasuk NEGATIF terhadap konten atau dokter/influencer kesehatan dalam video tersebut.

DEFINISI (penting, baca pelan-pelan):
- NEGATIF = komentar dengan nada negatif (keluhan, kekecewaan, sarkasme, kritik) yang DITUJUKAN ke konten videonya atau ke dokter/influencernya.
- BUKAN NEGATIF = semua yang lain, termasuk: curhat/keluhan pribadi (soal penyakit, hidup, pengalaman sendiri) yang TIDAK menyerang/mengkritik dokter atau kontennya, pertanyaan biasa, pujian, komentar netral/lucu.
- Keluhan pribadi (misal \"capek banget\", \"gerd saya kambuh\", \"8 tahun saya begini\") itu BUKAN NEGATIF meskipun nadanya sedih/kesal, SELAMA tidak menyindir dokter/kontennya.

Contoh dari data yang sudah pernah dilabel manusia:
{fewshot}

Sekarang nilai komentar baru ini:
\"{comment}\"

Jawab PERSIS dengan format berikut, tanpa tambahan apapun:
LABEL: negatif ATAU LABEL: bukan_negatif
ALASAN: <1 kalimat singkat>"""


def build_fewshot_block(neg_examples, nonneg_examples):
    lines = []
    for t in neg_examples:
        lines.append(f'- "{t}" -> LABEL: negatif')
    for t in nonneg_examples:
        lines.append(f'- "{t}" -> LABEL: bukan_negatif')
    return '\n'.join(lines)


def parse_judge_response(text: str):
    label_match = re.search(r'LABEL:\s*(negatif|bukan_negatif)', text, re.IGNORECASE)
    alasan_match = re.search(r'ALASAN:\s*(.+)', text, re.IGNORECASE)
    label = label_match.group(1).lower() if label_match else 'parse_gagal'
    alasan = alasan_match.group(1).strip() if alasan_match else text.strip()[:200]
    return label, alasan


def judge_comment(comment: str, fewshot_block: str, model: str = JUDGE_MODEL, retries: int = 3) -> tuple:
    prompt = JUDGE_PROMPT_TEMPLATE.format(fewshot=fewshot_block, comment=comment.replace('"', "'"))
    for attempt in range(retries):
        try:
            r = requests.post(f'{OLLAMA_HOST}/api/generate',
                               json={'model': model, 'prompt': prompt, 'stream': False,
                                     'options': {'temperature': 0.0}},
                               timeout=60)
            r.raise_for_status()
            raw = r.json()['response']
            return parse_judge_response(raw)
        except Exception as e:
            if attempt == retries - 1:
                return 'error', str(e)[:200]
            time.sleep(1.5 * (attempt + 1))

## Layer 1 — Semantic search pakai `bge-m3`

Embed 49 komentar negatif yang sudah dilabel (acuan), lalu embed semua komentar di `comments.csv`
(di-cache per teks unik). Skor = kemiripan maksimum ke salah satu dari 49 acuan itu.

Kalau cell ini di-run ulang (misal karena sebelumnya ke-interrupt), dia bakal lapor berapa yang
sudah ke-cache dan cuma lanjut embed sisanya.

In [10]:
labeled = pd.read_csv(LABELED_PATH)
ref_negatif_texts = labeled.loc[labeled['label_name'] == 'negatif', 'text_model'].dropna().unique().tolist()
ref_nonneg_texts  = labeled.loc[labeled['label_name'] != 'negatif', 'text_model'].dropna().tolist()
print(f'Acuan negatif: {len(ref_negatif_texts)} komentar unik')

print('Embedding acuan negatif...')
ref_result = embed_with_cache(ref_negatif_texts, EMBED_CACHE_REF_PATH, 'Embedding acuan negatif')
ref_embeds = np.vstack([ref_result[t] for t in ref_negatif_texts])

raw = pd.read_csv(RAW_PATH)
raw = raw.dropna(subset=['comment_text']).copy()
raw['comment_text'] = raw['comment_text'].astype(str).str.strip()
raw = raw[raw['comment_text'].str.len() > 0].reset_index(drop=True)

# exclude komentar yang sudah ada di data berlabel (hindari duplikat)
already_labeled = set(labeled['comment_text_clean'].dropna().astype(str).str.strip())
before = len(raw)
raw = raw[~raw['comment_text'].isin(already_labeled)].reset_index(drop=True)
print(f'Baris mentah: {before} -> {len(raw)} setelah exclude yang sudah dilabel')

Acuan negatif: 49 komentar unik
Embedding acuan negatif...
  cache ditemukan: 49 embedding tersimpan sebelumnya (embed_cache_acuan_negatif.csv)
  sudah di-embed sebelumnya: 49 dari 49
  perlu embed baru         : 0
Baris mentah: 27073 -> 25139 setelah exclude yang sudah dilabel


In [11]:
unique_texts = raw['comment_text'].unique().tolist()
print(f'Teks unik yang perlu di-embed: {len(unique_texts)}')

raw_result = embed_with_cache(unique_texts, EMBED_CACHE_RAW_PATH, 'Embedding comments.csv')

valid_texts = [t for t in unique_texts if t in raw_result]
skipped = len(unique_texts) - len(valid_texts)
if skipped:
    print(f'PERHATIAN: {skipped} teks gagal di-embed terus-menerus, dilewati (skor 0).')

emb_matrix = np.vstack([raw_result[t] for t in valid_texts])
sims = cosine_sim_matrix(emb_matrix, ref_embeds)
max_sim = sims.max(axis=1)
score_map = dict(zip(valid_texts, max_sim))

raw['skor_kemiripan_negatif'] = raw['comment_text'].map(score_map).fillna(0.0)
raw.sort_values('skor_kemiripan_negatif', ascending=False, inplace=True)
raw.reset_index(drop=True, inplace=True)

layer1 = raw.head(N_CANDIDATES).copy()
layer1.to_csv(LAYER1_PATH, index=False)
print(f'Layer 1 selesai: {len(layer1)} kandidat disimpan ke {LAYER1_PATH}')
print(layer1[['comment_text', 'skor_kemiripan_negatif']].head(10).to_string())

Teks unik yang perlu di-embed: 23854
  cache ditemukan: 4642 embedding tersimpan sebelumnya (embed_cache_comments.csv)
  sudah di-embed sebelumnya: 4642 dari 23854
  perlu embed baru         : 19212


Embedding comments.csv:   0%|          | 0/19212 [00:02<?, ?it/s]


KeyboardInterrupt: 

## Layer 2 — LLM judge pakai `qwen2.5:7b`

Few-shot-nya diambil langsung dari komentar yang SUDAH dilabel manusia (bukan contoh karangan), supaya LLM mengikuti standar yang sama kayak proses labeling kamu.

Ada **checkpoint**: kalau proses ini ke-interrupt di tengah jalan, jalankan ulang cell ini -- baris yang sudah di-judge nggak akan diulang.

In [12]:
# Cell tambahan: pastikan `layer1` siap buat Layer 2, pakai cache yang SUDAH ADA aja (skip sisanya)
SKIP_REMAINING_EMBEDDING = globals().get('SKIP_REMAINING_EMBEDDING', True)

if 'layer1' in globals() and layer1 is not None:
    print(f'`layer1` sudah ada di memori ({len(layer1)} kandidat), lanjut ke Layer 2.')
elif LAYER1_PATH.exists():
    layer1 = pd.read_csv(LAYER1_PATH)
    print(f'layer1_kandidat_embedding.csv sudah ada -> load langsung ({len(layer1)} kandidat), TIDAK embed ulang.')
else:
    import json as _json
    unique_texts = raw['comment_text'].unique().tolist()
    print(f'Teks unik di comments.csv: {len(unique_texts)}')

    # Baca cache embedding yang SUDAH ADA langsung dari csv (read-only, tidak nulis apa-apa)
    cache_map = {}
    if EMBED_CACHE_RAW_PATH.exists():
        _cache_df = pd.read_csv(EMBED_CACHE_RAW_PATH)
        for _, _row in _cache_df.iterrows():
            try:
                cache_map[_row['text']] = np.array(_json.loads(_row['embedding_json']), dtype=np.float32)
            except Exception:
                continue
    print(f'  embedding yang sudah ke-cache: {len(cache_map)} baris')

    valid_texts = [t for t in unique_texts if t in cache_map]
    skipped = len(unique_texts) - len(valid_texts)
    print(f'  dari {len(unique_texts)} teks unik: {len(valid_texts)} sudah ke-embed, {skipped} di-skip (SKIP_REMAINING_EMBEDDING aktif)')

    emb_matrix = np.vstack([cache_map[t] for t in valid_texts])
    sims = cosine_sim_matrix(emb_matrix, ref_embeds)
    max_sim = sims.max(axis=1)
    score_map = dict(zip(valid_texts, max_sim))
    raw['skor_kemiripan_negatif'] = raw['comment_text'].map(score_map).fillna(0.0)
    raw.sort_values('skor_kemiripan_negatif', ascending=False, inplace=True)
    raw.reset_index(drop=True, inplace=True)
    layer1 = raw.head(N_CANDIDATES).copy()
    layer1.to_csv(LAYER1_PATH, index=False)
    print(f'Layer 1 selesai: {len(layer1)} kandidat disimpan ke {LAYER1_PATH}')

print(layer1[['comment_text', 'skor_kemiripan_negatif']].head(10).to_string())

`layer1` sudah ada di memori (5000 kandidat), lanjut ke Layer 2.
          comment_text  skor_kemiripan_negatif
0              praktek                0.867674
1  marahin dok marahin                0.851336
2          praktek kak                0.830743
3        praktek kakaa                0.808973
4  Klo gini sehat gak🤣                0.792402
5         benerr bgttt                0.774907
6      sama2 gak sehat                0.771879
7   gak bs gènggam dok                0.755717
8   ga ada uang ny dok                0.748262
9            ga enak 😭                0.743773


In [13]:
rng = np.random.default_rng(SEED)
fewshot_neg    = rng.choice(ref_negatif_texts, size=min(N_FEWSHOT_EACH, len(ref_negatif_texts)), replace=False).tolist()
fewshot_nonneg = rng.choice(ref_nonneg_texts,  size=min(N_FEWSHOT_EACH, len(ref_nonneg_texts)),  replace=False).tolist()
fewshot_block  = build_fewshot_block(fewshot_neg, fewshot_nonneg)
print(fewshot_block)

- "harta tahta postingan tidak tersedia gerakan sayang kamu warna kulit cerah" -> LABEL: negatif
- "praktek ya praktekkk" -> LABEL: negatif
- "liat yt nya sampek abis kak, jangan cuman asal komen di tiktok yg setengah2, ini pasien nya di yt disebut kok nama dokternya dan rs nya, dan dia juga bawak bukti ctscen nya!!" -> LABEL: negatif
- "dok dines dimanaaa?biar bisa dihindarin wajah menangis keras" -> LABEL: negatif
- "apa cerita pagi ini" -> LABEL: bukan_negatif
- "geli dan ngilu bu bidd" -> LABEL: bukan_negatif
- "anaku sering kena bronco penomoni berulang bitu kenapa dok sampe sekarang malah kena asma alergi" -> LABEL: bukan_negatif
- "kaget banget gapernah sakit tbtb sakit h-1 wajah menangis keras wajah menangis keras wajah menangis keras wajah menangis keras wajah menangis keras" -> LABEL: bukan_negatif


In [14]:
# resume dari checkpoint kalau ada
COOLDOWN_EVERY   = 30   # jeda tiap N komentar
COOLDOWN_SECONDS = 0   # lama jedanya (detik)

if FINAL_PATH.exists():
    done_df = pd.read_csv(FINAL_PATH)
    done_ids = set(done_df['comment_id'])
    print(f'Checkpoint ditemukan: {len(done_df)} komentar sudah di-judge sebelumnya, lanjut dari situ.')
else:
    done_df = pd.DataFrame()
    done_ids = set()

todo = layer1[~layer1['comment_id'].isin(done_ids)].copy()
print(f'Sisa yang perlu di-judge: {len(todo)} dari {len(layer1)}')

results = []
t0 = time.time()
for i, row in enumerate(tqdm(todo.itertuples(index=False), total=len(todo), desc='LLM judge')):
    label, alasan = judge_comment(row.comment_text, fewshot_block)
    results.append({
        'comment_id': row.comment_id, 'username': row.username, 'video_id': row.video_id,
        'comment_text': row.comment_text, 'skor_kemiripan_negatif': row.skor_kemiripan_negatif,
        'judge_label': label, 'judge_alasan': alasan,
    })
    if (i + 1) % CHECKPOINT_EVERY == 0:
        partial = pd.concat([done_df, pd.DataFrame(results)], ignore_index=True)
        partial.to_csv(FINAL_PATH, index=False)

    if (i + 1) % COOLDOWN_EVERY == 0:
        tqdm.write(f'  jeda {COOLDOWN_SECONDS}s buat dinginin laptop... ({i+1}/{len(todo)})')
        time.sleep(COOLDOWN_SECONDS)

final_df = pd.concat([done_df, pd.DataFrame(results)], ignore_index=True)
final_df.to_csv(FINAL_PATH, index=False)
print(f'\nSelesai dalam {(time.time()-t0)/60:.1f} menit. Total ter-judge: {len(final_df)}')
print(final_df['judge_label'].value_counts().to_string())

Checkpoint ditemukan: 4800 komentar sudah di-judge sebelumnya, lanjut dari situ.
Sisa yang perlu di-judge: 200 dari 5000


LLM judge:  15%|█▌        | 30/200 [01:52<10:37,  3.75s/it]

  jeda 0s buat dinginin laptop... (30/200)


LLM judge:  30%|███       | 60/200 [03:44<08:38,  3.71s/it]

  jeda 0s buat dinginin laptop... (60/200)


LLM judge:  45%|████▌     | 90/200 [05:34<06:16,  3.42s/it]

... (5 outputs left out)

LLM judge:  90%|█████████ | 180/200 [11:06<01:12,  3.62s/it]

  jeda 0s buat dinginin laptop... (180/200)


LLM judge: 100%|██████████| 200/200 [12:19<00:00,  3.70s/it]


Selesai dalam 12.3 menit. Total ter-judge: 5000
judge_label
bukan_negatif    2559
error            2100
negatif           341


## Ringkasan hasil

Urutkan: yang di-label `negatif` oleh LLM paling atas (sesuai skor kemiripan), baru sisanya -- biar langsung ketemu yang paling prioritas buat direview manual.

In [15]:
final_df['urutan'] = final_df['judge_label'].map({'negatif': 0, 'bukan_negatif': 1}).fillna(2)
final_df.sort_values(['urutan', 'skor_kemiripan_negatif'], ascending=[True, False], inplace=True)
final_df.drop(columns='urutan', inplace=True)
final_df.to_csv(FINAL_PATH, index=False)

n_negatif = (final_df['judge_label'] == 'negatif').sum()
n_parse_gagal = final_df['judge_label'].isin(['parse_gagal', 'error']).sum()
print(f'LLM bilang NEGATIF  : {n_negatif} dari {len(final_df)} kandidat')
print(f'Gagal parse/error   : {n_parse_gagal}  (cek manual juga, mungkin false negative)')
print(f'\nFile final: {FINAL_PATH}')
print('Kolom: comment_id, username, video_id, comment_text, skor_kemiripan_negatif, judge_label, judge_alasan')
print('\n-> Review manual dimulai dari baris yang judge_label = negatif, lalu cek juga parse_gagal/error.')

LLM bilang NEGATIF  : 341 dari 5000 kandidat
Gagal parse/error   : 2100  (cek manual juga, mungkin false negative)

File final: c:\Users\nesha\Tugas_akhir\seleksi-influencer\01_pengumpulan_data\06_data_negatif_tambahan\hasil\kandidat_final_untuk_review.csv
Kolom: comment_id, username, video_id, comment_text, skor_kemiripan_negatif, judge_label, judge_alasan

-> Review manual dimulai dari baris yang judge_label = negatif, lalu cek juga parse_gagal/error.
